# 10 — Bronze load
Landing Parquet (written by ADF) → Bronze Delta. Appends only batches that are not yet in Bronze,
adds the audit columns, and never modifies source values.

In [ ]:
%run ./_common

In [ ]:
from pyspark.errors import AnalysisException

results = []
for table, cfg in TABLES.items():
    target = fq("bronze", table)
    path = f"{LANDING_ROOT}/sqldb/{table}"
    try:
        raw = spark.read.option("mergeSchema", "true").parquet(path)   # ingest_date / batch_id come from the folder names
    except AnalysisException as exc:
        print(f"{table}: nothing in landing ({str(exc).splitlines()[0][:80]})")
        continue

    if spark.catalog.tableExists(target):
        loaded = [r["batch_id"] for r in spark.table(target).select("batch_id").distinct().collect()]
        new = raw.where(~F.col("batch_id").cast("string").isin(loaded)) if loaded else raw
    else:
        new = raw

    new = (new
           .withColumn("batch_id", F.col("batch_id").cast("string"))
           .withColumn("source_record_id", F.concat_ws("|", *[F.col(k).cast("string") for k in cfg["pk"]]))
           .withColumn("source_updated_at",
                       F.col(SOURCE_UPDATED_COL[table]).cast("timestamp") if table in SOURCE_UPDATED_COL
                       else F.lit(None).cast("timestamp"))
           .withColumn("operation", F.lit(OPERATION[cfg["kind"]]))
           .withColumn("_bronze_loaded_at", F.current_timestamp()))
    new = new.cache()
    n = new.count()
    if n == 0:
        print(f"{table}: no new batches")
        new.unpersist()
        continue
    new.write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable(target)
    batches = [r["batch_id"] for r in new.select("batch_id").distinct().collect()]
    results.append((table, n, len(batches)))
    print(f"{table}: appended {n} rows from {len(batches)} batch(es)")
    new.unpersist()

In [ ]:
display(spark.createDataFrame(results, "table string, rows long, batches long")) if results else print("no new data")